In [1]:
import openai
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
client = openai.OpenAI()

client

In [4]:
from datetime import datetime
def get_delivery_date(order_id: str) -> datetime:
    """
    사용자의 id를 입력 받아서 배송일자를 조회하는 함수
    """
    return datetime.today().strftime("%Y-%m-%d")

get_delivery_date("a")

'2026-05-14'

In [7]:
# LLM에서 사용할 함수 tool 생성
delivery_tool = {
    "type": "function", # 도구의 타입을 function으로 설정하여, 함수 호출 기능 제공을 명시
    "function":{
        # 함수의 목적과 사용 사례를 설명
        #  챗봇이 언제 이 함수를 호출해야 하는지 명확히 알 수 있도록 설명을 포함.
        "name": "get_delivery_date", # 함수의 이름을 정의. 이 이름은 챗봇이 개발자에게 호출해라고 알려주는 역할
        "description": """
          고객의 주문에 대한 배송 날짜를 확인합니다.
          예를 들어, 고객이 "내 패키지가 어디에 있나요?"라고 물을 때 이 함수를 호출하세요.
        """,

        # 함수를 호출하기 위해 필요한 파라미터에 대한 설명
        "parameters": {
            "type": "object", # 파라미터의 타입
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "고객의 주문 ID"
                }
            },
            # 함수 호출 시 반드시 제공되어야 하는 필수 파라미터
            "required": ["order_id"],

            # 함수에 정의되지 않은 파라미터를 받을지 여부(*args, **kwargs 허용 여부)
            "additionalProperties": False
        }
    }
}

tools = [ delivery_tool ]

In [8]:
messages = [
    {"role": "system", "content": "당신은 도움이 되는 고객 지원 어시스턴스입니다. 제공된 도구를 사용하여 사용자를 지원하세요."},
    {"role": "user", "content": "안녕하세요, 제 주문의 배송 날짜를 알려주실 수 있나요?"},
    
]

response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=messages,
    tools=tools # 지원 도구 설정
)
print(response.choices[0].message)

ChatCompletionMessage(content='주문 ID를 알려주시면 배송 날짜를 확인해드리겠습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [10]:
messages = [
    {"role": "system", "content": "당신은 도움이 되는 고객 지원 어시스턴스입니다. 제공된 도구를 사용하여 사용자를 지원하세요."},
    {"role": "user", "content": "안녕하세요, 제 주문의 배송 날짜를 알려주실 수 있나요?"},
    {"role": "assistant", "content": '주문 ID를 알려주시면 배송 날짜를 확인해 드리겠습니다. 주문 ID를 제공해주세요.'},
    {"role": "user", "content": "제 생각엔 order12345인 것 같아요."}
]

response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=messages,
    tools=tools # 지원 도구 설정
)
print(response.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_sXMFVS3v6TqpoVocSjVIePJO', function=Function(arguments='{"order_id":"order12345"}', name='get_delivery_date'), type='function')]
